# Cooperative Solution of Multi-Regional Transmission Expansion Horizontal Investment Coordination Problem

### This notebook contains codes for the three zone test case (consisting of the IEEE 14 nodes, IEEE 30 nodes, and a 5 node test system for each of the zones) under the assumption that the regional transmission planners all cooperate to achieve the maximum value of the social surplus of the broader region

#### The first code block loads the different package dependencies to create the virtual environment in Julia

In [ ]:
# activate julia environment in a controlled way
julia_environment = "../central_control_PyJuMP/Julia_src/activate.jl"
include(julia_environment)

In [ ]:
using XLSX
using CSV
using GLPK
using DataFrames
using JuMP

#### The next code block reads the different files corresponding to shared candidate lines, internal candidate lines, shared existing lines, internal existing lines, generators, and load

In [ ]:
shared_cand = DataFrame(XLSX.readtable("../Input_Data/CandLine.csv", "Taul1", header=true)) #Dataframe of shared candidate lines
int_cand =  DataFrame(XLSX.readtable("../Input_Data/CandLineInt.csv", "Taul1", header=true)) #Dataframe of internal candidate lines
shared_ex =  DataFrame(XLSX.readtable("../Input_Data/SharedEline.csv", "Taul1", header=true)) #Dataframe of shared existing lines
int_ex = DataFrame(XLSX.readtable("../Input_Data/Tran.csv", "Taul1", header=true)) #Dataframe of internal existing lines
gen =  DataFrame(XLSX.readtable("../Input_Data/Gen.csv" , "Taul1", header=true)) #Dataframe of generators
load =  DataFrame(XLSX.readtable("../Input_Data/Load.csv", "Taul1", header=true)) #Dataframe of loads
scen_prob = DataFrame(CSV.File("../Input_Data/Scenario_Probability.csv", header=true)) #Dataframe of scenario probabilities
zone_summary = DataFrame(CSV.File("../Input_Data/Zone_Summary.csv", header=true)) #Dataframe of region-number of nodes
##An example of the load dataframe can be found as follows

#### The following code-section constructs several DataFrames for each zone or region, for loads within a particular zone, generators within a particular zone, shared candidate lines originating from or terminating within a particular zone, internal candidate lines within a particular zone, shared existing lines originating from or terminating within a particular zone, internal existing lines within a particular zone, marginal costs of generators within a particular zone, and two binary vectors for each zone, one for checking which all shared candidate lines belong to a particular zone and the other for checking which all shared existing lines belong to a particular zone.

In [ ]:
l(i,s) = load[load.zoneNum .== i, [1,2,s+2]] # load within zone i
g(i) = gen[gen.zoneNum .== i, :]   # generators within zone i
shared_c(i) = vcat(shared_cand[shared_cand.nodeZone1 .== i,:] , shared_cand[shared_cand.nodeZone2 .== i, :]) #shared candidate lines within zone i
int_c(i) = int_cand[int_cand.zoneNum .== i, :]   # number of internal candidate lines within zone i
shared_e(i) = vcat(shared_ex[shared_ex.nodeZone1 .== i,:] , shared_ex[shared_ex.nodeZone2 .== i, :]) #shared existing lines within zone i
int_e(i) =int_ex[int_ex.zoneNum .== i, :]       # internal existing lines within zone i
MC(i) = (g(i).C2 .* (g(i).PgMax .^ 2) .+ g(i).C1 .* g(i).PgMax .- g(i).C2 .*(g(i).PgMin .^ 2) .- g(i).C1 .* g(i).PgMin) ./ (g(i).PgMax .- g(i).PgMin) #Marginal cost of generators within zone i
bin_c(i) = (shared_cand.nodeZone1 .== i) + (shared_cand.nodeZone2 .== i) # A binary vector through which we can check if the shared candidate lines belong to zone i
bin_e(i) = (shared_ex.nodeZone1 .== i) + (shared_ex.nodeZone2 .== i) # A binary vector through which we can check if the shared existing lines belong to zone i
scen_weight(s) = scen_prob.scen_weight[scen_prob.scenario .== s,:]

#### Constructing the optimization model with a specified solver

In [ ]:
Mod3 = Model(GLPK.Optimizer) #use different solver e.g. IPOPT, GLPK

In [ ]:
zone_number = 3;

In [ ]:
@variable(Mod3,0 <= gen_var_1[1:14, 1:nrow(scen_prob)]) # generation at node 1
@variable(Mod3,0 <= gen_var_2[1:30, 1:nrow(scen_prob)]) # generation at node 2
@variable(Mod3,0 <= gen_var_3[1:5, 1:nrow(scen_prob)]) # generation at node 3

@variable(Mod3,shared_line_decision_var_1[1:nrow(shared_cand)], Bin) #Decision variable for shared candidate lines
@variable(Mod3,shared_line_decision_var_2[1:nrow(shared_cand)], Bin) #Decision variable for shared candidate lines
@variable(Mod3,shared_line_decision_var_3[1:nrow(shared_cand)], Bin) #Decision variable for shared candidate lines

@variable(Mod3,int_line_decision_var_1[1:nrow(int_c(1))], Bin) #Decision variable for internal candidate lines of zone 1 (Binary)
@variable(Mod3,int_line_decision_var_2[1:nrow(int_c(2))], Bin) #Decision variable for internal candidate lines of zone 2 (Binary)
@variable(Mod3,int_line_decision_var_3[1:nrow(int_c(3))], Bin) #Decision variable for internal candidate lines of zone 3 (Binary)

@variable(Mod3,shared_cand_flow[1:nrow(shared_cand),1:nrow(scen_prob)])  #Power flowing on shared candidate lines 

@variable(Mod3,int_cand_flow_1[1:nrow(int_c(1)),1:nrow(scen_prob)]) #Power flowing on internal candidate lines of zone 1 
@variable(Mod3,int_cand_flow_2[1:nrow(int_c(2)),1:nrow(scen_prob)]) #Power flowing on internal candidate lines of zone 2
@variable(Mod3,int_cand_flow_3[1:nrow(int_c(3)),1:nrow(scen_prob)]) #Power flowing on internal candidate lines of zone 3

@variable(Mod3, 0 <= node_voltage_phase_angle[i=1:nrow(zone_summary),j=1:zone_summary.Nodes_Total[i], 1:nrow(scen_prob)]<= 2*pi)

@variable(Mod3,shared_ex_flow[1:nrow(shared_ex),1:nrow(scen_prob)])  #Power flowing on shared existing shared lines

@variable(Mod3,int_ex_flow_1[1:nrow(int_e(1)),1:nrow(scen_prob)])  #Power flowing on internal existing lines of zone 1 
@variable(Mod3,int_ex_flow_2[1:nrow(int_e(2)),1:nrow(scen_prob)])  #Power flowing on internal existing lines of zone 2
@variable(Mod3,int_ex_flow_3[1:nrow(int_e(3)),1:nrow(scen_prob)])  #Power flowing on internal existing lines of zone 3

In [ ]:
@expression(Mod3,total_cost,0)

In [ ]:
@expression(Mod3, gen_costNS1[n=1:14,s=1:nrow(scen_prob)], (gen_var_1[n,s] .* sum((g(1).gNodeID .== n) .* MC(1))))

In [ ]:
@expression(Mod3, gen_cost_prob_weightedS1[s=1:nrow(scen_prob)], (scen_weight(s)[1]).*sum(gen_costNS1[n,s] for n in 1:14))

In [ ]:
@expression(Mod3, gen_cost_expected1, sum(gen_cost_prob_weightedS1[s] for s in 1:nrow(scen_prob)))
Mod3[:total_cost] += gen_cost_expected1

In [ ]:
@expression(Mod3, gen_costNS2[n=1:30,s=1:nrow(scen_prob)], (gen_var_2[n,s] .* sum((g(2).gNodeID .== n) .* MC(2))))

In [ ]:
@expression(Mod3, gen_cost_prob_weightedS2[s=1:nrow(scen_prob)], (scen_weight(s)[1]).*sum(gen_costNS2[n,s] for n in 1:30))

In [ ]:
@expression(Mod3, gen_cost_expected2, sum(gen_cost_prob_weightedS2[s] for s in 1:nrow(scen_prob)))
Mod3[:total_cost] += gen_cost_expected2

In [ ]:
@expression(Mod3, gen_costNS3[n=1:5,s=1:nrow(scen_prob)], (gen_var_3[n,s] .* sum((g(3).gNodeID .== n) .* MC(3))))

In [ ]:
@expression(Mod3, gen_cost_prob_weightedS3[s=1:nrow(scen_prob)], (scen_weight(s)[1]).*sum(gen_costNS3[n,s] for n in 1:5))

In [ ]:
@expression(Mod3, gen_cost_expected3, sum(gen_cost_prob_weightedS3[s] for s in 1:nrow(scen_prob)))
Mod3[:total_cost] += gen_cost_expected3

In [ ]:
@expression(Mod3, investment_cost , sum(bin_c(1)[c] .* shared_line_decision_var_1[c] .* shared_cand.costPerCap[c] .* shared_cand.interestRate[c] 
            .*((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c]) ./ (((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c])-1) for c in 1:nrow(shared_cand))
    .+ sum(bin_c(2)[c] .* shared_line_decision_var_2[c] .* shared_cand.costPerCap[c] .* shared_cand.interestRate[c] 
            .*((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c]) ./ (((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c])-1) for c in 1:nrow(shared_cand))
    .+ sum(bin_c(3)[c] .* shared_line_decision_var_3[c] .* shared_cand.costPerCap[c] .* shared_cand.interestRate[c] 
            .*((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c]) ./ (((1 + shared_cand.interestRate[c]) .^ shared_cand.lifeTime[c])-1) for c in 1:nrow(shared_cand))
    .+ sum(int_line_decision_var_1[c] .* int_c(1).costPerCap[c] .* int_c(1).interestRate[c] 
            .*((1 + int_c(1).interestRate[c]) .^ int_c(1).lifeTime[c]) ./ (((1 + int_c(1).interestRate[c]) .^ int_c(1).lifeTime[c])-1) for c in 1:nrow(int_c(1)))
    .+ sum(int_line_decision_var_2[c] .* int_c(2).costPerCap[c] .* int_c(2).interestRate[c] 
            .*((1 + int_c(2).interestRate[c]) .^ int_c(2).lifeTime[c]) ./ (((1 + int_c(2).interestRate[c]) .^ int_c(2).lifeTime[c])-1) for c in 1:nrow(int_c(2)))
    .+ sum(int_line_decision_var_3[c] .* int_c(3).costPerCap[c] .* int_c(3).interestRate[c] 
            .*((1 + int_c(3).interestRate[c]) .^ int_c(3).lifeTime[c]) ./ (((1 + int_c(3).interestRate[c]) .^ int_c(3).lifeTime[c])-1) for c in 1:nrow(int_c(3))))
Mod3[:total_cost]+=investment_cost

In [ ]:
#The first zone
for n in 1:14
    for s in 1:nrow(scen_prob)
        # Power balance constraint for each node
        @constraint(Mod3, sum(g(1).gNodeID .== n) .* gen_var_1[n,s] .+ sum(l(1,s)[:,3].* (l(1,s).lNodeID .== n))./100 .==
        sum((shared_cand.tNodeID1 .== n) .*bin_c(1) .* shared_cand_flow[:,s]) .- sum((shared_cand.tNodeID2 .== n) .* bin_c(1) .* shared_cand_flow[:,s]) .+
        sum((shared_ex.tNodeID1 .== n) .* bin_e(1) .* shared_ex_flow[:,s]) .- sum((shared_ex.tNodeID2 .== n) .* bin_e(1) .* shared_ex_flow[:,s]) .+
        sum((int_c(1).tNodeID1 .== n) .* int_cand_flow_1[:,s]) .- sum((int_c(1).tNodeID2 .== n) .* int_cand_flow_1[:,s]) .+
        sum((int_e(1).tNodeID1 .== n) .* int_ex_flow_1[:,s]) .- sum((int_e(1).tNodeID2 .== n) .* int_ex_flow_1[:,s]))
        #Lower limit for generation of each node
        @constraint(Mod3, sum(g(1).gNodeID .== n) .* gen_var_1[n,s] .<= sum((g(1).gNodeID .== n) .* g(1).PgMax)./100)
        #Upper limit for generation of each node
        @constraint(Mod3, sum((g(1).gNodeID .== n) .* g(1).PgMin)./100 .<= sum(g(1).gNodeID .== n) .* gen_var_1[n,s])
    end
end

In [ ]:
#The second zone
for n in 1:30
    for s in 1:nrow(scen_prob)
        # Power balance constraint for each node
        @constraint(Mod3, sum(g(2).gNodeID .== n) .* gen_var_2[n,s] .+ sum(l(2,s)[:,3] .* (l(2,s).lNodeID .== n))./100 .==
        sum((shared_cand.tNodeID1 .== n) .*bin_c(2) .* shared_cand_flow[:,s]) .- sum((shared_cand.tNodeID2 .== n) .* bin_c(2) .* shared_cand_flow[:,s]) .+
        sum((shared_ex.tNodeID1 .== n) .* bin_e(2) .* shared_ex_flow[:,s]) .- sum((shared_ex.tNodeID2 .== n) .* bin_e(2) .* shared_ex_flow[:,s]) .+
        sum((int_c(2).tNodeID1 .== n) .* int_cand_flow_2[:,s]) .- sum((int_c(2).tNodeID2 .== n) .* int_cand_flow_2[:,s]) .+
        sum((int_e(2).tNodeID1 .== n) .* int_ex_flow_2[:,s]) .- sum((int_e(2).tNodeID2 .== n) .* int_ex_flow_2[:,s]))
        #Lower limit for generation of each node
        @constraint(Mod3, sum(g(2).gNodeID .== n) .* gen_var_2[n,s] .<= sum((g(2).gNodeID .== n) .* g(2).PgMax)./100)
        #Upper limit for generation of each node
        @constraint(Mod3, sum((g(2).gNodeID .== n) .* g(2).PgMin)./100 .<= sum(g(2).gNodeID .== n) .* gen_var_2[n,s])
    end
end

In [ ]:
#The third zone
for n in 1:5
    for s in 1:nrow(scen_prob)
        # Power balance constraint for each node
        @constraint(Mod3, sum(g(3).gNodeID .== n) .* gen_var_3[n,s] .+ sum(l(3,s)[:,3] .* (l(3,s).lNodeID .== n))./100 .==
        sum((shared_cand.tNodeID1 .== n) .*bin_c(3) .* shared_cand_flow[:,s]) .- sum((shared_cand.tNodeID2 .== n) .* bin_c(3) .* shared_cand_flow[:,s]) .+
        sum((shared_ex.tNodeID1 .== n) .* bin_e(3) .* shared_ex_flow[:,s]) .- sum((shared_ex.tNodeID2 .== n) .* bin_e(3) .* shared_ex_flow[:,s]) .+
        sum((int_c(3).tNodeID1 .== n) .* int_cand_flow_3[:,s]) .- sum((int_c(3).tNodeID2 .== n) .* int_cand_flow_3[:,s]) .+
        sum((int_e(3).tNodeID1 .== n) .* int_ex_flow_3[:,s]) .- sum((int_e(3).tNodeID2 .== n) .* int_ex_flow_3[:,s]))
        #Lower limit for generation of each node
        @constraint(Mod3, sum(g(3).gNodeID .== n) .* gen_var_3[n,s] .<= sum((g(3).gNodeID .== n) .* g(3).PgMax)./100)
        #Upper limit for generation of each node
        @constraint(Mod3, sum((g(3).gNodeID .== n)./100 .* g(3).PgMin) .<= sum(g(3).gNodeID .== n) .* gen_var_3[n,s])
    end
end

In [ ]:
#Shared candidate lines
M = 10
for c in 1:nrow(shared_cand)
    for s in 1:nrow(scen_prob)
        @constraint(Mod3,-M .* (1 .- bin_c(1) .* shared_line_decision_var_1[c]) .<= bin_c(1) .* shared_cand_flow[c,s] .- bin_c(1) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])))
        @constraint(Mod3,-M .* (1 .- bin_c(2) .* shared_line_decision_var_2[c]) .<= bin_c(2) .* shared_cand_flow[c,s] .- bin_c(2) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])))
        @constraint(Mod3,-M .* (1 .- bin_c(3) .* shared_line_decision_var_3[c]) .<= bin_c(3) .* shared_cand_flow[c,s] .- bin_c(3) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])))
        
        @constraint(Mod3, bin_c(1) .* shared_cand_flow[c,s] .- bin_c(1) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])) .<= M .* (1 .- bin_c(1) .* shared_line_decision_var_1[c]))
        @constraint(Mod3, bin_c(2) .* shared_cand_flow[c,s] .- bin_c(2) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])) .<= M .* (1 .- bin_c(2) .* shared_line_decision_var_2[c]))
        @constraint(Mod3, bin_c(3) .* shared_cand_flow[c,s] .- bin_c(3) .* ((1 ./ shared_cand.reacT[c]) .* (node_voltage_phase_angle[shared_cand.nodeZone1[c],shared_cand.tNodeID1[c],s] .- node_voltage_phase_angle[shared_cand.nodeZone2[c],shared_cand.tNodeID2[c],s])) .<= M .* (1 .- bin_c(3) .* shared_line_decision_var_3[c]))
        #limiting the upper bound of power flow flowing within candidate shared lines
        @constraint(Mod3, bin_c(1) .* shared_cand_flow[c,s] .<= bin_c(1) .* shared_line_decision_var_1[c] .*shared_cand.ptMax[c]./100)
        @constraint(Mod3, bin_c(2) .* shared_cand_flow[c,s] .<= bin_c(2) .* shared_line_decision_var_2[c] .*shared_cand.ptMax[c]./100)
        @constraint(Mod3, bin_c(3) .* shared_cand_flow[c,s] .<= bin_c(3) .* shared_line_decision_var_3[c] .*shared_cand.ptMax[c]./100)
        #Limiting the lower bound of power flowing within the candidate shared lines
        @constraint(Mod3, -(shared_cand.ptMax[c] .* bin_c(1) .* shared_line_decision_var_1[c]./100) .<= bin_c(1) .* shared_cand_flow[c,s])
        @constraint(Mod3, -(shared_cand.ptMax[c] .* bin_c(2) .* shared_line_decision_var_2[c]./100) .<= bin_c(2) .* shared_cand_flow[c,s])
        @constraint(Mod3, -(shared_cand.ptMax[c] .* bin_c(3) .* shared_line_decision_var_3[c]./100) .<= bin_c(3) .* shared_cand_flow[c,s])
    end
 
end

In [ ]:
#Shared existing lines
for h in 1:nrow(shared_ex)
    for s in 1:nrow(scen_prob)
        @constraint(Mod3, shared_ex_flow[h,s] .== (1 ./ shared_ex.reacT[h]) .* (node_voltage_phase_angle[shared_ex.nodeZone1[h],shared_ex.tNodeID1[h],s] .- node_voltage_phase_angle[shared_ex.nodeZone2[h],shared_ex.tNodeID2[h],s]))
        @constraint(Mod3, shared_ex_flow[h,s] .<= shared_ex. ptMax[h]./100)
        @constraint(Mod3, -shared_ex.ptMax[h]./100 .<= shared_ex_flow[h,s])
    end
end

In [ ]:
#Zone 1 internal candidate lines
for c in 1:nrow(int_c(1))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3,-M .* (1 .- int_line_decision_var_1[c]) .<= int_cand_flow_1[c,s] .- ((1 ./ int_c(1).reacT[c]) .* (node_voltage_phase_angle[int_c(1).zoneNum[c],int_c(1).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(1).zoneNum[c],int_c(1).tNodeID2[c],s])))
        @constraint(Mod3, int_cand_flow_1[c,s] .- ((1 ./ int_c(1).reacT[c]) .* (node_voltage_phase_angle[int_c(1).zoneNum[c],int_c(1).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(1).zoneNum[c],int_c(1).tNodeID2[c],s])) .<= M .* (1 .- int_line_decision_var_1[c]))
        @constraint(Mod3, int_cand_flow_1[c,s] .<= int_line_decision_var_1[c] .* int_c(1).ptMax[c]./100)
        @constraint(Mod3, -(int_c(1).ptMax[c] .*int_line_decision_var_1[c]./100) .<= int_cand_flow_1[c,s])
    end
end

In [ ]:
#Zone 2 internal candidate lines
for c in 1:nrow(int_c(2))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3,-M .* (1 .- int_line_decision_var_2[c]) .<= int_cand_flow_2[c,s] .- ((1 ./ int_c(2).reacT[c]) .* (node_voltage_phase_angle[int_c(2).zoneNum[c],int_c(2).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(2).zoneNum[c],int_c(2).tNodeID2[c],s])))
        @constraint(Mod3, int_cand_flow_2[c,s] .- ((1 ./ int_c(2).reacT[c]) .* (node_voltage_phase_angle[int_c(2).zoneNum[c],int_c(2).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(2).zoneNum[c],int_c(2).tNodeID2[c],s])) .<= M .* (1 .- int_line_decision_var_2[c]))
        @constraint(Mod3, int_cand_flow_2[c,s] .<= int_line_decision_var_2[c] .* int_c(2).ptMax[c]./100)
        @constraint(Mod3, -(int_c(2).ptMax[c] .*int_line_decision_var_2[c]./100) .<= int_cand_flow_2[c,s])
    end
end

In [ ]:
#Zone 3 internal candidate lines
for c in 1:nrow(int_c(3))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3,-M .* (1 .- int_line_decision_var_3[c]) .<= int_cand_flow_3[c,s] .- ((1 ./ int_c(3).reacT[c]) .* (node_voltage_phase_angle[int_c(3).zoneNum[c],int_c(3).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(3).zoneNum[c],int_c(3).tNodeID2[c],s])))
        @constraint(Mod3, int_cand_flow_3[c,s] .- ((1 ./ int_c(3).reacT[c]) .* (node_voltage_phase_angle[int_c(3).zoneNum[c],int_c(3).tNodeID1[c],s] .- node_voltage_phase_angle[int_c(3).zoneNum[c],int_c(3).tNodeID2[c],s])) .<= M .* (1 .- int_line_decision_var_3[c]))
        @constraint(Mod3, int_cand_flow_3[c,s] .<= int_line_decision_var_3[c] .* int_c(3).ptMax[c]./100)
        @constraint(Mod3, -(int_c(3).ptMax[c] .*int_line_decision_var_3[c]./100) .<= int_cand_flow_3[c,s])
    end
end

In [ ]:
#Zone 1 internal existing lines
for h in 1:nrow(int_e(1))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3, int_ex_flow_1[h,s] .== (1 ./ int_e(1).reacT[h]) .* (node_voltage_phase_angle[int_e(1).zoneNum[h],int_e(1).tNodeID1[h],s] .- node_voltage_phase_angle[int_e(1).zoneNum[h],int_e(1).tNodeID2[h],s]))
        @constraint(Mod3, int_ex_flow_1[h,s] .<= int_e(1).ptMax[h]./100)
        @constraint(Mod3, -(int_e(1).ptMax[h]./100) .<= int_ex_flow_1[h,s])
    end
end

In [ ]:
#Zone 2 internal existing lines
for h in 1:nrow(int_e(2))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3, int_ex_flow_2[h,s] .== (1 ./ int_e(2).reacT[h]) .* (node_voltage_phase_angle[int_e(2).zoneNum[h],int_e(2).tNodeID1[h],s] .- node_voltage_phase_angle[int_e(2).zoneNum[h],int_e(2).tNodeID2[h],s]))
        @constraint(Mod3, int_ex_flow_2[h,s] .<= int_e(2).ptMax[h]./100)
        @constraint(Mod3, -(int_e(2).ptMax[h]./100) .<= int_ex_flow_2[h,s])
    end
end

In [ ]:
#Zone 3 internal existing lines
for h in 1:nrow(int_e(3))
    for s in 1:nrow(scen_prob)
        @constraint(Mod3, int_ex_flow_3[h,s] .== (1 ./ int_e(3).reacT[h]) .* (node_voltage_phase_angle[int_e(3).zoneNum[h],int_e(3).tNodeID1[h],s] .- node_voltage_phase_angle[int_e(3).zoneNum[h],int_e(3).tNodeID2[h],s]))
        @constraint(Mod3, int_ex_flow_3[h,s] .<= int_e(3).ptMax[h]./100)
        @constraint(Mod3, -(int_e(3).ptMax[h]./100) .<= int_ex_flow_3[h,s])
    end
end

In [ ]:
@objective(Mod3, Min, Mod3[:total_cost])
optimize!(Mod3)

In [ ]:
if termination_status(Mod3) == MOI.OPTIMAL
    shared_line_decision_1 = value.(shared_line_decision_var_1)
    shared_line_decision_2 = value.(shared_line_decision_var_2)
    shared_line_decision_3 = value.(shared_line_decision_var_3)
    shared_cand_power_flow = (value.(shared_cand_flow)) .* 100
    node_voltage_phase_angle = value.(node_voltage_phase_angle)
    shared_ex_power_flow = (value.(shared_ex_flow)) .* 100
    int_line_decision_1 = value.(int_line_decision_var_1)
    int_line_decision_2 = value.(int_line_decision_var_2)
    int_line_decision_3 = value.(int_line_decision_var_3)
    int_cand_line_flow_1 = (value.(int_cand_flow_1)) .* 100
    int_cand_line_flow_2 = (value.(int_cand_flow_2)) .* 100
    int_cand_line_flow_3 = (value.(int_cand_flow_3)) .* 100
    generation_1 = (value.(gen_var_1)) .* 100
    generation_2 = (value.(gen_var_2)) .* 100
    generation_3 = (value.(gen_var_3)) .* 100
    internal_existing_line_flow_1 = (value.(int_ex_flow_1)) .* 100
    internal_existing_line_flow_2 = (value.(int_ex_flow_2)) .* 100
    internal_existing_line_flow_3 = (value.(int_ex_flow_3)) .* 100
    obj_value = objective_value(Mod3)
end

In [ ]:
shared_line_decision_1

In [ ]:
shared_line_decision_2

In [ ]:
shared_line_decision_3

In [ ]:
shared_cand_power_flow

In [ ]:
node_voltage_phase_angle

In [ ]:
shared_ex_power_flow

In [ ]:
int_line_decision_1

In [ ]:
int_line_decision_2

In [ ]:
int_line_decision_3

In [ ]:
shared_cand

In [ ]:
int_cand_line_flow_1

In [ ]:
int_cand_line_flow_2

In [ ]:
int_cand_line_flow_3

In [ ]:
internal_existing_line_flow_1

In [ ]:
internal_existing_line_flow_2

In [ ]:
internal_existing_line_flow_3

In [ ]:
generation_1

In [ ]:
generation_2

In [ ]:
generation_3

In [ ]:
obj_value